In [1]:
import pandas as pd
import gzip
import numpy as np
from pathlib import Path

In [30]:
def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield eval(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

df = getDF('reviews_Clothing_Shoes_and_Jewelry_5.json.gz')

In [31]:
new_df = df[["reviewerID", "asin", "overall", "unixReviewTime"]].copy()
new_df.columns = ["user_id", "item_id", "rating", "timestamp"]
new_df["rating"] = 1

In [32]:
new_df["user_id"], unique_user_ids = pd.factorize(new_df["user_id"])
new_df["item_id"], unique_item_ids = pd.factorize(new_df["item_id"])
new_df["user_id"] += 1
new_df["item_id"] += 1

In [33]:
new_df.to_csv("clothing.csv", index=False)

In [2]:
df = pd.DataFrame({"a": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], "b": [1, 1, 2, 2, 3, 3, 3, 3, 3, 3]})

In [3]:
df.sort_values("b", ascending=False)

,a,b
4,4,3
5,5,3
8,8,3
9,9,3
7,7,3
6,6,3
2,2,2
3,3,2
1,1,1
0,0,1


In [40]:
df.sort_values("b", ascending=False)

,a,b
4,4,3
5,5,3
8,8,3
9,9,3
7,7,3
6,6,3
2,2,2
3,3,2
1,1,1
0,0,1


In [ ]:
df_sorted = new_df.sort_values(by="timestamp")

test_treshold = int(len(df_sorted) * 0.98)
val_treshold = int(len(df_sorted) * 0.96)

train_val = df_sorted.head(test_treshold)
warm_test = df_sorted.tail(len(df_sorted) - test_treshold)
test = warm_test.loc[warm_test.groupby("user_id")["timestamp"].idxmax()]
warm_t = warm_test[~warm_test.index.isin(test.index)]


train = df_sorted.head(val_treshold)
test_val = df_sorted.tail(len(df_sorted) - val_treshold)
warm_val = test_val[~test_val.index.isin(warm_test.index)]
val = warm_val.loc[warm_val.groupby("user_id")["timestamp"].idxmax()]
warm_v = warm_val[~warm_val.index.isin(val.index)]


train = pd.concat([train, warm_v])

In [6]:
print("Train len: ", len(train))
print("Train users: ", len(train["user_id"].unique()))
print("Val len: ", len(val))
print("Val users: ", len(val["user_id"].unique()))
print("Test len: ", len(test))
print("Test users: ", len(test["user_id"].unique()))
print("Warm val len: ", len(warm_v))
print("Warm val users: ", len(warm_v["user_id"].unique()))
print("Warm test len: ", len(warm_t))
print("Warm test users: ", len(warm_t["user_id"].unique()))
print(
    "test_val intersection users: ",
    np.intersect1d(val["user_id"].unique(), test["user_id"].unique()).shape[0],
)

Train len:  192812
Train users:  22236
Val len:  1719
Val users:  1719
Test len:  1685
Test users:  1685
Warm val len:  2251
Warm val users:  805
Warm test len:  2286
Warm test users:  775
test_val intersection users:  432


In [7]:
dataset_folder = Path("data2/Beauty4")
dataset_folder.mkdir()

In [8]:
user_items = train.groupby("user_id")["item_id"].apply(list).to_dict()

with open(dataset_folder / "train.txt", "w") as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [9]:
train_warm_v = pd.concat([train, warm_v], ignore_index=True)
user_items = warm_v.groupby("user_id")["item_id"].apply(list).to_dict()

with open(dataset_folder / "warm_val.txt", "w") as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [10]:
filtered_train = train_warm_v[train_warm_v["user_id"].isin(val["user_id"])]
train_val = pd.concat([filtered_train, val], ignore_index=True)

user_items = train_val.groupby("user_id")["item_id"].apply(list).to_dict()
with open(dataset_folder / "val.txt", "w") as f:
    for user_id, items in user_items.items():
        if len(items) > 1:
            f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [11]:
unique_indices = val["user_id"].unique()
with open(dataset_folder / "val_users.txt", "w") as f:
    for index in unique_indices:
        f.write(f"{index}\n")

# Чтение чисел из файла и сохранение их в список
with open(dataset_folder / "val_users.txt", "r") as f:
    index_list = [int(line.strip()) for line in f]

In [12]:
filtered_train = df_sorted[df_sorted["user_id"].isin(test["user_id"])]


user_items = filtered_train.groupby("user_id")["item_id"].apply(list).to_dict()
with open(dataset_folder / "test.txt", "w") as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [13]:
user_items = df_sorted.groupby("user_id")["item_id"].apply(list).to_dict()

with open(dataset_folder / "all_data.txt", "w") as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")